In [ ]:
import torch
import pandas as pd
import copy
from torch.utils.data import DataLoader
from tqdm import tqdm
from pathlib import Path
from maw.datasets.landsat import LandsatDataset
from maw.datasets.sentinel import SentinelDataset
from maw.training.metrics import compute_metrics
from maw.models.resnet import build_resnet50
from importlib import import_module

In [ ]:
# Model Params
BATCH_SIZE = 32          # batch size for inference
N_WORKERS = 9            # workers for DataLoader
PREFETCH_FACTOR = 2      # prefetch factor for DataLoader
PIN_MEMORY = True

MODEL_CONFIGS = {        # model config, model checkpoint stem
    "sentinel": ("maw.configs.sentinel_resnet50", "SENTINEL-BASELINE-LR0.001-WD0.001-DP0.3"),
    "landsat": ("maw.configs.landsat_resnet50", "LANDSAT-BASELINE-LR0.001-WD0.001-DP0.3")
}

In [ ]:
# Paths to relevant files
HOME_DIR = Path.home() / "Projects/mapping-asia-wealth"
PREDICTIONS_OUTDIR = HOME_DIR / "data-clean/outputs/predictions/"       # path to save model predictions
EMBEDDINGS_OUTDIR = HOME_DIR / "data-raw/embeddings/training"           # path to save model embeddings

In [ ]:
PREDICTIONS_OUTDIR.mkdir(exist_ok=True, parents=True)
EMBEDDINGS_OUTDIR.mkdir(exist_ok=True, parents=True)

## Experiment Set-up

We aim to predict a household asset index from a 224 x 224 px image patch. Each patch has six spectral bands -- Red, Green, Blue, Near Infrared (NIR), Short-wave Infrared I (SWIR1), and Short-wave Infrared II (SWIR2). For each survey enumeration year, we build a three-year median composite (centered on that year) of all cloud-free Tier 1 images from Landsat 5, 7, 8, and 9 in the USGS/NASA Landsat archive on Google Earth Engine. To reduce the influence of outliers, we clip each band to its empirical 2nd and 98th percentiles (computed across the entire dataset) and then apply the USGS-recommended scaling factors so that reflectance values fall within the standard input range.

Our target variable is either an Absolute Wealth Index defined as the first principal component of asset-ownership survey questions. To maintain consistency, we pool all clusters from DHS5 onward and compute one global PCA using only the subset of asset questions common across those rounds. We define a secondary Relative Wealth Index based on the Demographic and Health Surveys (DHS) Wealth Index, which we average and normalised for each survey independently.  Surveys conducted before DHS 5 are excluded because they contain too few overlapping asset-related questions, which would unduly narrow our index definition.


In [ ]:
# Plot Sentinel Imagery
s2_mod_path, _ = MODEL_CONFIGS["sentinel"]
s2_mod = import_module(s2_mod_path)
s2 = SentinelDataset(obs_csv=s2_mod.cfg.labels_csv, root_dir=s2_mod.cfg.images_root, label_name=s2_mod.cfg.label_name,
                         transform=s2_mod.cfg.transforms_val.to("cpu"))
s2.plot_random_16()

In [ ]:
# Plot Landsat Imagery
ls_mod_path, _ = MODEL_CONFIGS["landsat"]
ls_mod = import_module(ls_mod_path)
ls = SentinelDataset(obs_csv=ls_mod.cfg.labels_csv, root_dir=ls_mod.cfg.images_root, label_name=ls_mod.cfg.label_name,
                         transform=ls_mod.cfg.transforms_val.to("cpu"))
ls.plot_random_16()

## 5-fold Spatial Cross-Validation
We divide the dataset into five geographically independent folds—each a set of non-overlapping observation clusters—stratified by country to maintain balance. Clusters are assigned to folds via a greedy algorithm (i.e. iteratively placing the next largest cluster into the fold with the fewest total observations) to minimize size imbalance and avoid spatial leakage. We then perform 5-fold cross-validation (CV): in each iteration, one fold serves as the held-out (validation) set, and the model is trained on the other four folds. After each round, we generate out-of-fold predictions on the held-out fold.

To evaluate overall performance, we pool all out-of-fold predictions across the five iterations and compute our metrics (e.g., R^2, RMSE, MAE) on this aggregate. This ensures every observation is validated exactly once, yielding an unbiased estimate of generalization across countries.


In [ ]:
for imagery_type in ["sentinel", "landsat"]:

    # Get model configs and checkpoint stem
    config_name, ckpt_stem = MODEL_CONFIGS[imagery_type]
    mod = import_module(config_name)

    # Set up transforms
    transforms = mod.cfg.transforms_val.to(mod.cfg.device)

    # Setup Dataloader
    if mod.cfg.imagery_type == "sentinel":
        dataset = SentinelDataset(obs_csv=mod.cfg.labels_csv, root_dir=mod.cfg.images_root, label_name=mod.cfg.label_name)
    elif mod.cfg.imagery_type == "landsat":
        dataset = LandsatDataset(obs_csv=mod.cfg.labels_csv, root_dir=mod.cfg.images_root, label_name=mod.cfg.label_name)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=N_WORKERS, prefetch_factor=PREFETCH_FACTOR, 
                       pin_memory=PIN_MEMORY)

    model_embeddings_list = []
    model_predictions_list = []
    for held_out in range(1, mod.cfg.n_folds + 1):

        # Load best-performing checkpoint for the held_out
        ckpt = f"{ckpt_stem}-SPLIT{held_out}-FINAL.pt"
        model = build_resnet50(hidden_dim=mod.cfg.fc_hidden, weights=mod.cfg.initial_weights).to(mod.cfg.device)
        model.load_state_dict(torch.load(mod.cfg.checkpoint_dir / ckpt, map_location=mod.cfg.device))
        embed_dim = model.fc[0].in_features
        
        head = copy.deepcopy(model.fc)                               # split model into backbone and head
        model.fc = torch.nn.Identity()
        backbone = model
        backbone.eval()
        head.eval()

        feats_list = []
        preds_list = []
        ids_list = []
        labels_list = []
        with torch.no_grad():
            for batch in tqdm(loader, desc=f"Est. Performance on Fold {held_out}", leave=True):
                x = transforms(batch["image"].to(mod.cfg.device))
                feats = torch.flatten(backbone(x), start_dim=1)     # extract features from backbone
                feats_list.append(feats.cpu())
                preds = head(feats)                                 # get CNN predictions
                preds_list.append(preds.cpu())
                ids_list.append(batch["patch_id"])
                labels_list.append(batch["label"])

        # Save embeddings and predictions
        meta = {
            "patch_id": [id for id_tup in ids_list for id in id_tup],
            "model_backbone": "resnet50-cnn",
            "model_backbone_name": ckpt,
            "model_backbone_heldoutset": held_out,
        }

        embed_df = pd.DataFrame(torch.cat(feats_list, dim=0).numpy(), columns=[f"X{i}" for i in range(embed_dim)]).assign(**meta)
        model_embeddings_list.append(embed_df)

        pred_df = dataset.obs_table.copy()
        pred_df["labels"] = torch.cat(labels_list, dim=0).numpy().flatten()
        pred_df["outputs"] = torch.cat(preds_list, dim=0).numpy().flatten()
        pred_df = pred_df.assign(**meta)
        model_predictions_list.append(pred_df)

        mae, rmse, r2 = compute_metrics(y_true=pred_df['labels'].to_numpy(), y_pred=pred_df['outputs'].to_numpy())
        print({"mae": mae, "rmse": rmse, "r2": r2})


    # Save embeddings and predictions
    embeddings_filename = f"{mod.cfg.imagery_type}-resnet50-training-embeddings.parquet"
    train_embeddings = pd.concat(model_embeddings_list).reset_index(drop=True)
    train_embeddings.to_parquet(EMBEDDINGS_OUTDIR / embeddings_filename)

    predictions_filename = f"{mod.cfg.imagery_type}-resnet50-training-predictions.parquet"
    train_predictions = pd.concat(model_predictions_list).reset_index(drop=True)
    train_predictions.to_parquet(PREDICTIONS_OUTDIR / predictions_filename)
